# 07 · Generalization: Qwen2.5-3B and MATH-500

* **3B recipe transfer**: retrain the verifier on `Qwen2.5-3B-Instruct`
  features (layers 18, 27, 36), 1,200-question pool, 400 GSM8K test questions.
  Baseline is majority voting with log-prob tie-break (89.75%).
* **Frozen-weight transfer**: map 3B features into the 1.5B-Instruct feature
  space with ridge regression (alpha = 10) and apply a frozen 1.5B verifier.
* **MATH-500**: `Qwen2.5-Math-1.5B-Instruct`, 3,000 MATH training problems as
  the pool, answers compared with `math-verify`.

Set `RUN_CAPTURE = True` to regenerate (GPU, about 1.5 h for 3B and about 10 h
for MATH); otherwise the cached draws are used.

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gpu,gbt,math] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
RUN_CAPTURE = False
if RUN_CAPTURE:
    from lvpp.capture import ChatCapture, ChatConfig
    from lvpp.answers import extract_chat_numeric, extract_boxed
    from lvpp.data import gsm8k, math500, math_train
    os.makedirs(f"{CACHE}/novelty", exist_ok=True); os.makedirs(f"{CACHE}/pending", exist_ok=True)
    ChatCapture(ChatConfig(model="Qwen/Qwen2.5-1.5B-Instruct")).build_cache(
        "1.5B pool", gsm8k("train"), extract_chat_numeric, PATHS["x15_pool"], limit=1200)
    cap3 = ChatCapture(ChatConfig(model="Qwen/Qwen2.5-3B-Instruct"))
    cap3.build_cache("3B pool", gsm8k("train"), extract_chat_numeric, PATHS["x3_pool"], limit=1200)
    cap3.build_cache("3B test", gsm8k("test"), extract_chat_numeric, PATHS["x3_test"], limit=400)
    capm = ChatCapture(ChatConfig(model="Qwen/Qwen2.5-Math-1.5B-Instruct", max_new=1536, qbatch=4))
    capm.build_cache("MATH pool", math_train(), extract_boxed, PATHS["math_pool"], limit=3000)
    capm.build_cache("MATH test", math500(), extract_boxed, PATHS["math_test"])

In [ ]:
#@title 3B: recipe transfer and frozen-weight transfer
from sklearn.linear_model import Ridge
from lvpp.caches import load_dict_B, dict_B_as_pool
from lvpp.selection import selection_hits
from lvpp.verifier import fit_verifier
rows = {}
P3, Z3, y3 = dict_B_as_pool(PATHS["x3_pool"]); t3 = load_dict_B(PATHS["x3_test"])
accs, aucs = [], []
for s in [0, 1, 2]:
    sc = fit_verifier(P3, Z3, y3, seed=s).score(t3.P, t3.Z)
    accs.append(100 * selection_hits(t3, "weighted", scores=sc).mean()); aucs.append(roc_auc_score(t3.y.numpy(), sc))
rows["3B majority (log-prob tie-break)"] = 100 * selection_hits(t3, "majority_lp").mean()
rows["3B retrained verifier"] = float(np.mean(accs)); rows["3B retrained AUC"] = float(np.mean(aucs))
P15, Z15, y15 = dict_B_as_pool(PATHS["x15_pool"])
v15 = fit_verifier(P15, Z15, y15, seed=0)
n = min(len(P15), len(P3))
ridge = Ridge(alpha=10.0).fit(P3[:n].numpy(), P15[:n].numpy())
sc = v15.score(torch.tensor(ridge.predict(t3.P.numpy())).float(), t3.Z)
rows["frozen 1.5B via ridge, accuracy"] = 100 * selection_hits(t3, "weighted", scores=sc).mean()
rows["frozen 1.5B via ridge, AUC"] = roc_auc_score(t3.y.numpy(), sc)
EXPECTED = {"3B majority (log-prob tie-break)": 89.75, "3B retrained verifier": 91.25, "3B retrained AUC": 0.8875,
            "frozen 1.5B via ridge, accuracy": 88.25, "frozen 1.5B via ridge, AUC": 0.732}
report(rows, EXPECTED, list(EXPECTED))

In [ ]:
#@title MATH-500 in-domain (3 seeds, K = 2 and 8)
from lvpp.answers import make_equiv
eq = make_equiv(True)
Pm, Zm, ym = dict_B_as_pool(PATHS["math_pool"], equiv=eq); tm = load_dict_B(PATHS["math_test"], equiv=eq)
scs = [fit_verifier(Pm, Zm, ym, seed=s).score(tm.P, tm.Z) for s in [0, 1, 2]]
for K in [2, 8]:
    maj = 100 * selection_hits(tm, "majority", K=K).mean(); lp = 100 * selection_hits(tm, "majority_lp", K=K).mean()
    ver = np.mean([100 * selection_hits(tm, "weighted", scores=s, K=K).mean() for s in scs])
    print(f"K={K}: majority {maj:.2f} | log-prob tie-break {lp:.2f} | LVPP {ver:.2f}")
print("in-domain AUC:", round(float(np.mean([roc_auc_score(tm.y.numpy(), s) for s in scs])), 4))
print("original run: K=8 log-prob tie-break 78.40 -> 78.33; K=2 gain 0.0 at majority 74.4; AUC 0.892")